# Tutorial 3: Your First Packet Classification

**Difficulty:** Beginner  
**Time Required:** 2-3 hours  
**Prerequisites:** Basic Python, PyTorch fundamentals  

## 📚 Learning Objectives

By completing this tutorial, you will:
- ✅ Load and preprocess network packet data
- ✅ Convert packet bytes to image representations
- ✅ Fine-tune a Vision Transformer for packet classification
- ✅ Evaluate model performance with metrics and visualizations
- ✅ Visualize model predictions and attention maps
- ✅ Deploy your trained model for inference

---

## 🚀 Step 1: Environment Setup

Let's start by importing all necessary libraries and setting up our environment for reproducible results.

In [ ]:
# Core libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Transformers and ML libraries
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure matplotlib for better plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📊 Step 2: Loading and Understanding Packet Data

For this tutorial, we'll create synthetic packet data that mimics real network traffic patterns. In practice, you would load real packet captures (PCAP files) or network monitoring data.

In [ ]:
def create_sample_data(n_samples=1000, packet_size=1024):
    """
    Create synthetic packet data for demonstration.
    
    Args:
        n_samples: Number of packet samples to generate
        packet_size: Size of each packet in bytes
    
    Returns:
        packets: List of packet byte arrays
        labels: List of labels (0=benign, 1=malicious)
    """
    packets = []
    labels = []
    
    print(f"🔄 Generating {n_samples} synthetic packets...")
    
    for i in tqdm(range(n_samples), desc="Creating packets"):
        if i < n_samples // 2:
            # Benign packets - more structured patterns
            # Simulate protocol headers and normal data patterns
            header = np.random.randint(0, 128, size=64, dtype=np.uint8)  # Protocol headers
            payload = np.random.randint(32, 127, size=packet_size-64, dtype=np.uint8)  # ASCII-like payload
            packet = np.concatenate([header, payload])
            labels.append(0)  # Benign
        else:
            # Malicious packets - more random/encrypted patterns
            # Simulate encrypted or obfuscated malicious traffic
            packet = np.random.randint(0, 256, size=packet_size, dtype=np.uint8)
            # Add some suspicious patterns
            if i % 10 == 0:  # Every 10th malicious packet has repeated patterns
                pattern = np.random.randint(200, 256, size=32, dtype=np.uint8)
                packet[100:132] = pattern  # Inject suspicious pattern
            labels.append(1)  # Malicious
        
        packets.append(bytes(packet))
    
    return packets, labels

# Generate sample data
packets, labels = create_sample_data(1000)

print(f"\n📈 Dataset Statistics:")
print(f"   Total packets: {len(packets)}")
print(f"   Benign packets: {labels.count(0)} ({labels.count(0)/len(labels)*100:.1f}%)")
print(f"   Malicious packets: {labels.count(1)} ({labels.count(1)/len(labels)*100:.1f}%)")
print(f"   Packet size: {len(packets[0])} bytes")

Let's examine some packet characteristics to understand our data better:

In [ ]:
# Analyze packet byte distributions
def analyze_packet_statistics(packets, labels, n_samples=5):
    """
    Analyze statistical properties of packet data.
    """
    benign_packets = [packets[i] for i in range(len(packets)) if labels[i] == 0]
    malicious_packets = [packets[i] for i in range(len(packets)) if labels[i] == 1]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Byte value histograms
    benign_bytes = np.concatenate([np.frombuffer(p, dtype=np.uint8) for p in benign_packets[:n_samples]])
    malicious_bytes = np.concatenate([np.frombuffer(p, dtype=np.uint8) for p in malicious_packets[:n_samples]])
    
    axes[0,0].hist(benign_bytes, bins=50, alpha=0.7, label='Benign', color='green')
    axes[0,0].hist(malicious_bytes, bins=50, alpha=0.7, label='Malicious', color='red')
    axes[0,0].set_title('Byte Value Distribution')
    axes[0,0].set_xlabel('Byte Value')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].legend()
    
    # Entropy analysis
    def calculate_entropy(data):
        _, counts = np.unique(data, return_counts=True)
        probabilities = counts / len(data)
        return -np.sum(probabilities * np.log2(probabilities + 1e-10))
    
    benign_entropies = [calculate_entropy(np.frombuffer(p, dtype=np.uint8)) for p in benign_packets[:20]]
    malicious_entropies = [calculate_entropy(np.frombuffer(p, dtype=np.uint8)) for p in malicious_packets[:20]]
    
    axes[0,1].boxplot([benign_entropies, malicious_entropies], labels=['Benign', 'Malicious'])
    axes[0,1].set_title('Packet Entropy Distribution')
    axes[0,1].set_ylabel('Entropy (bits)')
    
    # Raw packet visualization
    sample_benign = np.frombuffer(benign_packets[0], dtype=np.uint8)[:256]
    sample_malicious = np.frombuffer(malicious_packets[0], dtype=np.uint8)[:256]
    
    axes[1,0].plot(sample_benign, color='green', alpha=0.7, label='Benign')
    axes[1,0].set_title('Sample Benign Packet (First 256 bytes)')
    axes[1,0].set_xlabel('Byte Position')
    axes[1,0].set_ylabel('Byte Value')
    
    axes[1,1].plot(sample_malicious, color='red', alpha=0.7, label='Malicious')
    axes[1,1].set_title('Sample Malicious Packet (First 256 bytes)')
    axes[1,1].set_xlabel('Byte Position')
    axes[1,1].set_ylabel('Byte Value')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\n📊 Packet Analysis Results:")
    print(f"   Benign entropy: {np.mean(benign_entropies):.2f} ± {np.std(benign_entropies):.2f}")
    print(f"   Malicious entropy: {np.mean(malicious_entropies):.2f} ± {np.std(malicious_entropies):.2f}")

analyze_packet_statistics(packets, labels)

## 🖼️ Step 3: Packet to Image Conversion

The key innovation of our approach is converting 1D packet byte sequences into 2D images that can be processed by Vision Transformers. We'll implement three different encoding methods.

In [ ]:
class PacketImageEncoder:
    """
    Convert packet bytes to image representations using different encoding strategies.
    """
    
    def __init__(self, image_size=224):
        self.image_size = image_size
        self.total_pixels = image_size * image_size
        print(f"🔧 Initialized PacketImageEncoder with {image_size}x{image_size} images ({self.total_pixels} pixels)")
    
    def _prepare_bytes(self, packet_bytes):
        """Prepare byte array with padding or truncation."""
        byte_array = np.frombuffer(packet_bytes, dtype=np.uint8)
        
        if len(byte_array) < self.total_pixels:
            # Pad with zeros
            byte_array = np.pad(byte_array, (0, self.total_pixels - len(byte_array)))
        elif len(byte_array) > self.total_pixels:
            # Truncate
            byte_array = byte_array[:self.total_pixels]
        
        return byte_array
    
    def encode_sequential(self, packet_bytes):
        """
        Sequential encoding: arrange bytes in row-major order.
        Simple but preserves byte order locality.
        """
        byte_array = self._prepare_bytes(packet_bytes)
        
        # Reshape to 2D image
        image = byte_array.reshape(self.image_size, self.image_size)
        
        # Convert to RGB (repeat grayscale values)
        image_rgb = np.stack([image, image, image], axis=-1)
        
        return Image.fromarray(image_rgb.astype(np.uint8))
    
    def encode_hilbert(self, packet_bytes):
        """
        Hilbert curve encoding: preserves spatial locality better than sequential.
        Bytes that are close in the packet remain close in the image.
        """
        try:
            from hilbertcurve.hilbertcurve import HilbertCurve
        except ImportError:
            print("⚠️  Warning: hilbertcurve package not installed. Using sequential encoding instead.")
            return self.encode_sequential(packet_bytes)
        
        byte_array = self._prepare_bytes(packet_bytes)
        
        # Create Hilbert curve (image_size must be power of 2)
        p = int(np.log2(self.image_size))
        if 2**p != self.image_size:
            print(f"⚠️  Warning: Image size {self.image_size} is not a power of 2. Using sequential encoding.")
            return self.encode_sequential(packet_bytes)
        
        hilbert_curve = HilbertCurve(p, 2)
        
        # Create image
        image = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
        
        for i, byte_val in enumerate(byte_array):
            if i >= self.total_pixels:
                break
            
            # Get 2D coordinates from Hilbert curve
            coords = hilbert_curve.point_from_distance(i)
            image[coords[0], coords[1]] = byte_val
        
        # Convert to RGB
        image_rgb = np.stack([image, image, image], axis=-1)
        
        return Image.fromarray(image_rgb)
    
    def encode_spiral(self, packet_bytes):
        """
        Spiral encoding: arrange bytes in spiral pattern from center outward.
        Emphasizes early packet bytes (headers) in the center.
        """
        byte_array = self._prepare_bytes(packet_bytes)
        image = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
        
        # Generate spiral coordinates starting from center
        x, y = self.image_size // 2, self.image_size // 2
        dx, dy = 0, -1
        
        for i, byte_val in enumerate(byte_array):
            if i >= self.total_pixels:
                break
            
            if 0 <= x < self.image_size and 0 <= y < self.image_size:
                image[y, x] = byte_val
            
            # Spiral movement logic
            if x == y or (x < 0 and x == -y) or (x > 0 and x == 1 - y):
                dx, dy = -dy, dx
            
            x, y = x + dx, y + dy
        
        # Convert to RGB
        image_rgb = np.stack([image, image, image], axis=-1)
        
        return Image.fromarray(image_rgb)
    
    def encode_advanced(self, packet_bytes):
        """
        Advanced encoding: use different color channels for different byte characteristics.
        R: Original bytes, G: Byte differences, B: Frequency patterns
        """
        byte_array = self._prepare_bytes(packet_bytes)
        
        # Reshape to 2D
        base_image = byte_array.reshape(self.image_size, self.image_size)
        
        # Red channel: original bytes
        r_channel = base_image
        
        # Green channel: byte differences (highlights patterns)
        g_channel = np.zeros_like(base_image)
        for i in range(1, len(byte_array)):
            row, col = divmod(i, self.image_size)
            if row < self.image_size and col < self.image_size:
                g_channel[row, col] = abs(int(byte_array[i]) - int(byte_array[i-1]))
        
        # Blue channel: local frequency patterns
        b_channel = np.zeros_like(base_image)
        window_size = 16
        for i in range(0, len(byte_array) - window_size, window_size):
            window = byte_array[i:i+window_size]
            entropy = len(np.unique(window)) / window_size * 255
            
            for j in range(window_size):
                pos = i + j
                row, col = divmod(pos, self.image_size)
                if row < self.image_size and col < self.image_size:
                    b_channel[row, col] = int(entropy)
        
        # Combine channels
        image_rgb = np.stack([r_channel, g_channel, b_channel], axis=-1)
        
        return Image.fromarray(image_rgb.astype(np.uint8))

# Create encoder instance
encoder = PacketImageEncoder(image_size=224)
print("✅ PacketImageEncoder ready!")

Let's visualize the different encoding methods with sample packets:

In [ ]:
# Visualize different encoding methods
def compare_encoding_methods(packets, labels, encoder, n_samples=2):
    """
    Compare different encoding methods visually.
    """
    # Get one benign and one malicious packet
    benign_idx = labels.index(0)
    malicious_idx = labels.index(1)
    
    sample_packets = [
        (packets[benign_idx], "Benign Packet", "green"),
        (packets[malicious_idx], "Malicious Packet", "red")
    ]
    
    encoding_methods = [
        ('Sequential', encoder.encode_sequential),
        ('Spiral', encoder.encode_spiral),
        ('Advanced', encoder.encode_advanced)
    ]
    
    fig, axes = plt.subplots(len(sample_packets), len(encoding_methods), 
                           figsize=(15, 8))
    
    for i, (packet, packet_type, color) in enumerate(sample_packets):
        for j, (method_name, method_func) in enumerate(encoding_methods):
            # Generate image
            img = method_func(packet)
            
            # Plot
            axes[i, j].imshow(img)
            axes[i, j].set_title(f'{method_name}\n{packet_type}', color=color, fontweight='bold')
            axes[i, j].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Packet Encoding Methods Comparison', y=1.02, fontsize=16, fontweight='bold')
    plt.show()
    
    print("\n🎨 Encoding Methods Explained:")
    print("   📊 Sequential: Linear arrangement, preserves byte order")
    print("   🌀 Spiral: Center-out arrangement, emphasizes packet headers")
    print("   🎯 Advanced: Multi-channel encoding with pattern analysis")

compare_encoding_methods(packets, labels, encoder)

## 🧠 Step 4: Preparing Data for Vision Transformer

Now we'll create PyTorch datasets and data loaders to prepare our packet images for training.

In [ ]:
class PacketImageDataset(Dataset):
    """
    PyTorch dataset for packet images with different encoding methods.
    """
    
    def __init__(self, packets, labels, encoder, processor, encoding_method='sequential'):
        self.packets = packets
        self.labels = labels
        self.encoder = encoder
        self.processor = processor
        self.encoding_method = encoding_method
        
        print(f"📦 Created dataset with {len(packets)} samples using {encoding_method} encoding")
    
    def __len__(self):
        return len(self.packets)
    
    def __getitem__(self, idx):
        # Convert packet to image
        packet = self.packets[idx]
        
        # Select encoding method
        if self.encoding_method == 'sequential':
            image = self.encoder.encode_sequential(packet)
        elif self.encoding_method == 'spiral':
            image = self.encoder.encode_spiral(packet)
        elif self.encoding_method == 'advanced':
            image = self.encoder.encode_advanced(packet)
        else:
            raise ValueError(f"Unknown encoding method: {self.encoding_method}")
        
        # Process image for ViT (normalize, resize, etc.)
        inputs = self.processor(images=image, return_tensors="pt")
        
        # Remove batch dimension added by processor
        pixel_values = inputs['pixel_values'].squeeze(0)
        
        # Convert label to tensor
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return {
            'pixel_values': pixel_values,
            'labels': label
        }

# Split data into train/validation/test sets
print("🔄 Splitting dataset...")

# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    packets, labels, test_size=0.2, random_state=42, stratify=labels
)

# Second split: separate train and validation (80% train, 20% val of remaining)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp  # 0.25 * 0.8 = 0.2 of total
)

print(f"\n📊 Dataset Split:")
print(f"   Training set: {len(X_train)} samples ({len(X_train)/len(packets)*100:.1f}%)")
print(f"   Validation set: {len(X_val)} samples ({len(X_val)/len(packets)*100:.1f}%)")
print(f"   Test set: {len(X_test)} samples ({len(X_test)/len(packets)*100:.1f}%)")

# Check class balance
for split_name, split_labels in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    benign_count = split_labels.count(0)
    malicious_count = split_labels.count(1)
    print(f"   {split_name}: {benign_count} benign, {malicious_count} malicious")

## 🤖 Step 5: Loading and Configuring Vision Transformer

We'll use a pre-trained Vision Transformer from Hugging Face and adapt it for our binary classification task.

In [ ]:
# Load pre-trained ViT model and processor
model_name = "google/vit-base-patch16-224"
print(f"🔄 Loading pre-trained ViT model: {model_name}")

# Load image processor (handles normalization, resizing, etc.)
processor = ViTImageProcessor.from_pretrained(model_name)
print(f"   ✅ Image processor loaded")

# Load model and adapt for binary classification
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=2,  # Binary classification: benign vs malicious
    ignore_mismatched_sizes=True  # Allow different classifier head size
)
print(f"   ✅ Model loaded and adapted for binary classification")

# Move model to device (GPU if available)
model = model.to(device)
print(f"   ✅ Model moved to {device}")

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1e6:.1f} MB")

In [ ]:
# Create datasets with sequential encoding (we'll compare methods later)
encoding_method = 'sequential'
print(f"🔄 Creating datasets with {encoding_method} encoding...")

train_dataset = PacketImageDataset(X_train, y_train, encoder, processor, encoding_method)
val_dataset = PacketImageDataset(X_val, y_val, encoder, processor, encoding_method)
test_dataset = PacketImageDataset(X_test, y_test, encoder, processor, encoding_method)

# Create data loaders
batch_size = 16  # Smaller batch size for memory efficiency
num_workers = 2  # Adjust based on your system

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True if device.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True if device.type == 'cuda' else False
)

print(f"\n📦 Data Loaders Created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")
print(f"   Batch size: {batch_size}")

## 🏋️ Step 6: Training Configuration

Let's set up the optimizer, loss function, and training parameters.

In [ ]:
# Training configuration
learning_rate = 5e-5  # Conservative learning rate for fine-tuning
num_epochs = 5
weight_decay = 0.01

# Optimizer: AdamW (Adam with weight decay)
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=learning_rate,
    weight_decay=weight_decay
)

# Loss function: CrossEntropyLoss for classification
criterion = nn.CrossEntropyLoss()

# Learning rate scheduler (optional)
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=2, gamma=0.5)  # Reduce LR by half every 2 epochs

print(f"🔧 Training Configuration:")
print(f"   Learning rate: {learning_rate}")
print(f"   Number of epochs: {num_epochs}")
print(f"   Weight decay: {weight_decay}")
print(f"   Optimizer: AdamW")
print(f"   Loss function: CrossEntropyLoss")
print(f"   LR scheduler: StepLR (reduce by 0.5 every 2 epochs)")

# Variables to track best model
best_val_accuracy = 0.0
best_model_path = 'best_packet_classifier.pth'

# Lists to store training history
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
learning_rates = []

## 🎯 Step 7: Training and Validation Functions

Let's implement the training and evaluation functions with progress tracking.

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device, epoch_num):
    """
    Train the model for one epoch.
    
    Returns:
        avg_loss: Average training loss
        accuracy: Training accuracy
    """
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    # Progress bar
    pbar = tqdm(dataloader, desc=f"Epoch {epoch_num} - Training")
    
    for batch_idx, batch in enumerate(pbar):
        # Move data to device
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        # Forward pass
        outputs = model(pixel_values)
        loss = criterion(outputs.logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping (optional, helps with stability)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Calculate metrics
        _, predicted = torch.max(outputs.logits, 1)
        total_loss += loss.item()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        # Update progress bar
        current_acc = correct / total
        pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{current_acc:.4f}'
        })
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device, phase="Validation"):
    """
    Evaluate the model on validation or test set.
    
    Returns:
        avg_loss: Average loss
        accuracy: Accuracy
        all_predictions: List of all predictions
        all_labels: List of all true labels
    """
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc=f"{phase}")
        
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            outputs = model(pixel_values)
            loss = criterion(outputs.logits, labels)
            
            # Get predictions and probabilities
            probabilities = torch.softmax(outputs.logits, dim=1)
            _, predicted = torch.max(outputs.logits, 1)
            
            # Accumulate metrics
            total_loss += loss.item()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            # Store predictions and labels
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            
            # Update progress bar
            current_acc = correct / total
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{current_acc:.4f}'
            })
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy, all_predictions, all_labels, all_probabilities

print("✅ Training and evaluation functions ready!")

## 🚀 Step 8: Model Training

Now let's train our Vision Transformer on the packet image data!

In [ ]:
print("🚀 Starting training...\n")
print("=" * 60)

for epoch in range(num_epochs):
    print(f"\n📅 Epoch {epoch + 1}/{num_epochs}")
    print("-" * 40)
    
    # Record current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    learning_rates.append(current_lr)
    print(f"Learning rate: {current_lr:.2e}")
    
    # Training phase
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, epoch + 1)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # Validation phase
    val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, device, "Validation")
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # Learning rate scheduler step
    scheduler.step()
    
    # Print epoch results
    print(f"\n📊 Epoch {epoch + 1} Results:")
    print(f"   Training   - Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"   Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
    
    # Save best model
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_acc,
            'val_loss': val_loss
        }, best_model_path)
        print(f"   💾 New best model saved! (Validation accuracy: {val_acc:.4f})")
    
    print("=" * 60)

print(f"\n🎉 Training completed!")
print(f"📈 Best validation accuracy: {best_val_accuracy:.4f} ({best_val_accuracy*100:.2f}%)")
print(f"💾 Best model saved to: {best_model_path}")

## 📊 Step 9: Training History Visualization

Let's visualize how our model performed during training.

In [ ]:
# Plot training history
def plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies, learning_rates):
    """
    Plot comprehensive training history.
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    epochs = range(1, len(train_losses) + 1)
    
    # Loss plot
    axes[0, 0].plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    axes[0, 0].plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy plot
    axes[0, 1].plot(epochs, [acc*100 for acc in train_accuracies], 'b-', label='Training Accuracy', linewidth=2)
    axes[0, 1].plot(epochs, [acc*100 for acc in val_accuracies], 'r-', label='Validation Accuracy', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Learning rate plot
    axes[1, 0].plot(epochs, learning_rates, 'g-', linewidth=2, marker='o')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].set_title('Learning Rate Schedule')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Overfitting analysis
    train_val_gap = [abs(t - v) for t, v in zip(train_accuracies, val_accuracies)]
    axes[1, 1].plot(epochs, train_val_gap, 'purple', linewidth=2, marker='s')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy Gap')
    axes[1, 1].set_title('Train-Validation Accuracy Gap\n(Lower is better)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final statistics
    print(f"\n📈 Training Summary:")
    print(f"   Final training accuracy: {train_accuracies[-1]:.4f} ({train_accuracies[-1]*100:.2f}%)")
    print(f"   Final validation accuracy: {val_accuracies[-1]:.4f} ({val_accuracies[-1]*100:.2f}%)")
    print(f"   Best validation accuracy: {max(val_accuracies):.4f} ({max(val_accuracies)*100:.2f}%)")
    print(f"   Final accuracy gap: {abs(train_accuracies[-1] - val_accuracies[-1]):.4f}")
    
    if abs(train_accuracies[-1] - val_accuracies[-1]) > 0.1:
        print("   ⚠️  Large accuracy gap suggests potential overfitting")
    else:
        print("   ✅ Good generalization (small accuracy gap)")

plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies, learning_rates)

## 🧪 Step 10: Model Evaluation on Test Set

Now let's load our best model and evaluate it on the test set to get unbiased performance metrics.

In [ ]:
# Load the best model
print("🔄 Loading best model...")
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded model from epoch {checkpoint['epoch']} with validation accuracy {checkpoint['val_accuracy']:.4f}")

# Evaluate on test set
print("\n🧪 Evaluating on test set...")
test_loss, test_acc, test_predictions, test_labels, test_probabilities = evaluate(
    model, test_loader, criterion, device, "Test Set"
)

print(f"\n🎯 Test Set Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Convert probabilities to numpy array for easier handling
test_probabilities = np.array(test_probabilities)

## 📈 Step 11: Detailed Performance Analysis

Let's create comprehensive visualizations and metrics for our model's performance.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, auc

def comprehensive_evaluation(true_labels, predictions, probabilities):
    """
    Comprehensive evaluation with multiple metrics and visualizations.
    """
    # Confusion Matrix
    cm = confusion_matrix(true_labels, predictions)
    
    # Classification Report
    report = classification_report(true_labels, predictions, 
                                 target_names=['Benign', 'Malicious'], 
                                 output_dict=True)
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(true_labels, probabilities[:, 1])
    roc_auc = auc(fpr, tpr)
    
    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(true_labels, probabilities[:, 1])
    pr_auc = auc(recall, precision)
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
                xticklabels=['Benign', 'Malicious'],
                yticklabels=['Benign', 'Malicious'])
    axes[0, 0].set_title('Confusion Matrix')
    axes[0, 0].set_ylabel('True Label')
    axes[0, 0].set_xlabel('Predicted Label')
    
    # Add percentage annotations
    for i in range(2):
        for j in range(2):
            percentage = cm[i, j] / cm.sum() * 100
            axes[0, 0].text(j+0.5, i+0.7, f'({percentage:.1f}%)', 
                          ha='center', va='center', fontsize=10, color='gray')
    
    # 2. ROC Curve
    axes[0, 1].plot(fpr, tpr, color='darkorange', lw=2, 
                   label=f'ROC Curve (AUC = {roc_auc:.3f})')
    axes[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', alpha=0.5)
    axes[0, 1].set_xlim([0.0, 1.0])
    axes[0, 1].set_ylim([0.0, 1.05])
    axes[0, 1].set_xlabel('False Positive Rate')
    axes[0, 1].set_ylabel('True Positive Rate')
    axes[0, 1].set_title('ROC Curve')
    axes[0, 1].legend(loc="lower right")
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Precision-Recall Curve
    axes[1, 0].plot(recall, precision, color='darkgreen', lw=2, 
                   label=f'PR Curve (AUC = {pr_auc:.3f})')
    axes[1, 0].set_xlim([0.0, 1.0])
    axes[1, 0].set_ylim([0.0, 1.05])
    axes[1, 0].set_xlabel('Recall')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Precision-Recall Curve')
    axes[1, 0].legend(loc="lower left")
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Prediction Confidence Distribution
    benign_confidences = probabilities[np.array(true_labels) == 0, 0]
    malicious_confidences = probabilities[np.array(true_labels) == 1, 1]
    
    axes[1, 1].hist(benign_confidences, bins=20, alpha=0.7, label='Benign', color='green')
    axes[1, 1].hist(malicious_confidences, bins=20, alpha=0.7, label='Malicious', color='red')
    axes[1, 1].set_xlabel('Confidence Score')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Prediction Confidence Distribution')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed metrics
    print("\n📊 Detailed Performance Metrics:")
    print("=" * 50)
    
    # Overall metrics
    accuracy = report['accuracy']
    print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"PR AUC: {pr_auc:.4f}")
    
    print("\nPer-Class Metrics:")
    print("-" * 30)
    for class_name in ['Benign', 'Malicious']:
        class_idx = 0 if class_name == 'Benign' else 1
        metrics = report[str(class_idx)]
        print(f"{class_name:>10}:")
        print(f"   Precision: {metrics['precision']:.4f}")
        print(f"   Recall:    {metrics['recall']:.4f}")
        print(f"   F1-Score:  {metrics['f1-score']:.4f}")
        print(f"   Support:   {int(metrics['support'])}")
    
    # Error analysis
    print("\nError Analysis:")
    print("-" * 20)
    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives:  {tn:3d} (Correctly identified benign)")
    print(f"False Positives: {fp:3d} (Benign classified as malicious)")
    print(f"False Negatives: {fn:3d} (Malicious classified as benign)")
    print(f"True Positives:  {tp:3d} (Correctly identified malicious)")
    
    # Security-focused metrics
    print("\nSecurity Metrics:")
    print("-" * 20)
    detection_rate = tp / (tp + fn) if (tp + fn) > 0 else 0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    print(f"Detection Rate (Sensitivity): {detection_rate:.4f} ({detection_rate*100:.2f}%)")
    print(f"False Alarm Rate:            {false_alarm_rate:.4f} ({false_alarm_rate*100:.2f}%)")
    
    return {
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'detection_rate': detection_rate,
        'false_alarm_rate': false_alarm_rate,
        'confusion_matrix': cm,
        'classification_report': report
    }

# Run comprehensive evaluation
evaluation_results = comprehensive_evaluation(test_labels, test_predictions, test_probabilities)

## 🔍 Step 12: Attention Visualization

One of the key advantages of Vision Transformers is their interpretability. Let's visualize what parts of the packet images our model is focusing on.

In [ ]:
def visualize_attention_maps(model, dataset, encoder, processor, device, n_samples=4):
    """
    Visualize attention maps for sample predictions.
    """
    model.eval()
    
    # Get some sample indices
    sample_indices = np.random.choice(len(dataset), n_samples, replace=False)
    
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 4*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i, idx in enumerate(sample_indices):
        # Get the sample
        sample = dataset[idx]
        pixel_values = sample['pixel_values'].unsqueeze(0).to(device)
        true_label = sample['labels'].item()
        
        # Get the original packet and create image
        if hasattr(dataset, 'packets'):
            packet = dataset.packets[idx]
            original_image = encoder.encode_sequential(packet)
        else:
            # If we can't access original packet, use the processed image
            # Convert normalized image back to displayable format
            img_array = pixel_values[0].cpu().numpy().transpose(1, 2, 0)
            img_array = (img_array * 0.5 + 0.5) * 255  # Denormalize
            original_image = Image.fromarray(img_array.astype(np.uint8))
        
        # Get model prediction and attention
        with torch.no_grad():
            outputs = model(pixel_values, output_attentions=True)
            prediction = torch.argmax(outputs.logits, dim=1).item()
            confidence = torch.softmax(outputs.logits, dim=1).max().item()
            
            # Get attention from the last layer, average over heads
            attentions = outputs.attentions[-1][0].mean(dim=0).cpu().numpy()
            
            # Get attention to CLS token (first token)
            cls_attention = attentions[0, 1:]  # Skip CLS token itself
            
            # Reshape to spatial dimensions (14x14 for 224x224 input with 16x16 patches)
            num_patches = int(np.sqrt(cls_attention.shape[0]))
            attention_map = cls_attention.reshape(num_patches, num_patches)
        
        # Plot original image
        axes[i, 0].imshow(original_image)
        axes[i, 0].set_title(f'Original Packet Image\nTrue: {"{}".format("Malicious" if true_label == 1 else "Benign")}')
        axes[i, 0].axis('off')
        
        # Plot attention map
        im = axes[i, 1].imshow(attention_map, cmap='hot', interpolation='bilinear')
        axes[i, 1].set_title(f'Attention Map\nPred: {"{}".format("Malicious" if prediction == 1 else "Benign")} ({confidence:.2%})')
        axes[i, 1].axis('off')
        plt.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)
        
        # Plot overlay
        axes[i, 2].imshow(original_image, alpha=0.7)
        axes[i, 2].imshow(attention_map, cmap='hot', alpha=0.5, interpolation='bilinear',
                         extent=[0, original_image.width, original_image.height, 0])
        axes[i, 2].set_title('Attention Overlay')
        axes[i, 2].axis('off')
        
        # Add correctness indicator
        correctness = "✅ Correct" if prediction == true_label else "❌ Incorrect"
        color = 'green' if prediction == true_label else 'red'
        axes[i, 2].text(0.02, 0.98, correctness, transform=axes[i, 2].transAxes,
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor=color, alpha=0.7),
                       fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.suptitle('Vision Transformer Attention Analysis', y=1.02, fontsize=16, fontweight='bold')
    plt.show()
    
    print("\n🔍 Attention Analysis:")
    print("   🔥 Hot regions (red/yellow) indicate high attention")
    print("   ❄️  Cool regions (blue/purple) indicate low attention")
    print("   🎯 The model focuses on different packet regions for classification")

# Visualize attention maps
print("🔍 Generating attention visualizations...")
visualize_attention_maps(model, test_dataset, encoder, processor, device, n_samples=3)

## 🚀 Step 13: Model Deployment Wrapper

Let's create a production-ready wrapper for easy deployment and inference.

In [ ]:
class PacketClassifier:
    """
    Production-ready packet classifier wrapper.
    """
    
    def __init__(self, model_path, model_name="google/vit-base-patch16-224", 
                 encoding_method='sequential', image_size=224):
        """
        Initialize the packet classifier.
        
        Args:
            model_path: Path to the trained model file
            model_name: Hugging Face model name
            encoding_method: Packet encoding method
            image_size: Size of packet images
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.encoder = PacketImageEncoder(image_size=image_size)
        self.processor = ViTImageProcessor.from_pretrained(model_name)
        self.encoding_method = encoding_method
        self.model_name = model_name
        
        # Load model
        self.model = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=2,
            ignore_mismatched_sizes=True
        )
        
        # Load trained weights
        if isinstance(model_path, str):
            checkpoint = torch.load(model_path, map_location=self.device)
            if 'model_state_dict' in checkpoint:
                self.model.load_state_dict(checkpoint['model_state_dict'])
            else:
                self.model.load_state_dict(checkpoint)
        
        self.model.to(self.device)
        self.model.eval()
        
        print(f"🤖 PacketClassifier initialized:")
        print(f"   Model: {model_name}")
        print(f"   Device: {self.device}")
        print(f"   Encoding: {encoding_method}")
        print(f"   Ready for inference! 🚀")
    
    def _encode_packet(self, packet_bytes):
        """Convert packet bytes to image."""
        if self.encoding_method == 'sequential':
            return self.encoder.encode_sequential(packet_bytes)
        elif self.encoding_method == 'spiral':
            return self.encoder.encode_spiral(packet_bytes)
        elif self.encoding_method == 'advanced':
            return self.encoder.encode_advanced(packet_bytes)
        else:
            raise ValueError(f"Unknown encoding method: {self.encoding_method}")
    
    def classify_packet(self, packet_bytes, return_attention=False):
        """
        Classify a single packet.
        
        Args:
            packet_bytes: Raw packet bytes
            return_attention: Whether to return attention weights
        
        Returns:
            Dictionary with classification results
        """
        # Convert to image
        image = self._encode_packet(packet_bytes)
        
        # Process image
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs['pixel_values'].to(self.device)
        
        # Get prediction
        with torch.no_grad():
            outputs = self.model(pixel_values, output_attentions=return_attention)
            probabilities = torch.softmax(outputs.logits, dim=1)
            prediction = torch.argmax(outputs.logits, dim=1).item()
            confidence = probabilities[0, prediction].item()
        
        result = {
            'prediction': 'malicious' if prediction == 1 else 'benign',
            'confidence': confidence,
            'probabilities': {
                'benign': probabilities[0, 0].item(),
                'malicious': probabilities[0, 1].item()
            },
            'risk_score': probabilities[0, 1].item()  # Malicious probability as risk score
        }
        
        # Add attention if requested
        if return_attention and outputs.attentions:
            attention = outputs.attentions[-1][0].mean(dim=0).cpu().numpy()
            cls_attention = attention[0, 1:]  # CLS token attention to patches
            num_patches = int(np.sqrt(cls_attention.shape[0]))
            result['attention_map'] = cls_attention.reshape(num_patches, num_patches)
        
        return result
    
    def classify_batch(self, packet_list, batch_size=32):
        """
        Classify multiple packets efficiently.
        
        Args:
            packet_list: List of packet bytes
            batch_size: Batch size for processing
        
        Returns:
            List of classification results
        """
        results = []
        
        for i in tqdm(range(0, len(packet_list), batch_size), desc="Classifying packets"):
            batch = packet_list[i:i+batch_size]
            
            # Convert batch to images
            images = [self._encode_packet(packet) for packet in batch]
            
            # Process batch
            inputs = self.processor(images=images, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(self.device)
            
            # Get predictions
            with torch.no_grad():
                outputs = self.model(pixel_values)
                probabilities = torch.softmax(outputs.logits, dim=1)
                predictions = torch.argmax(outputs.logits, dim=1)
            
            # Process results
            for j in range(len(batch)):
                pred = predictions[j].item()
                conf = probabilities[j, pred].item()
                
                results.append({
                    'prediction': 'malicious' if pred == 1 else 'benign',
                    'confidence': conf,
                    'probabilities': {
                        'benign': probabilities[j, 0].item(),
                        'malicious': probabilities[j, 1].item()
                    },
                    'risk_score': probabilities[j, 1].item()
                })
        
        return results
    
    def get_model_info(self):
        """Get information about the loaded model."""
        total_params = sum(p.numel() for p in self.model.parameters())
        return {
            'model_name': self.model_name,
            'encoding_method': self.encoding_method,
            'device': str(self.device),
            'total_parameters': total_params,
            'model_size_mb': total_params * 4 / 1e6
        }

# Create classifier instance
classifier = PacketClassifier(best_model_path, encoding_method='sequential')

# Display model info
model_info = classifier.get_model_info()
print(f"\n📋 Model Information:")
for key, value in model_info.items():
    print(f"   {key}: {value}")

## 🧪 Step 14: Testing the Deployment

Let's test our deployed model with some examples.

In [ ]:
# Test single packet classification
print("🧪 Testing single packet classification...\n")

# Test with a few packets from our test set
test_packets = X_test[:5]
test_labels_subset = y_test[:5]

for i, (packet, true_label) in enumerate(zip(test_packets, test_labels_subset)):
    result = classifier.classify_packet(packet, return_attention=False)
    
    true_class = 'malicious' if true_label == 1 else 'benign'
    correct = '✅' if result['prediction'] == true_class else '❌'
    
    print(f"Packet {i+1}: {correct}")
    print(f"   True class: {true_class}")
    print(f"   Predicted: {result['prediction']} (confidence: {result['confidence']:.2%})")
    print(f"   Risk score: {result['risk_score']:.3f}")
    print(f"   Probabilities: Benign={result['probabilities']['benign']:.3f}, "
          f"Malicious={result['probabilities']['malicious']:.3f}")
    print()

# Test batch classification
print("\n🚀 Testing batch classification...")
batch_results = classifier.classify_batch(X_test[:20], batch_size=8)

# Summary statistics
correct_predictions = sum(1 for i, result in enumerate(batch_results) 
                         if (result['prediction'] == 'malicious') == (y_test[i] == 1))
batch_accuracy = correct_predictions / len(batch_results)

print(f"   Batch size: {len(batch_results)}")
print(f"   Correct predictions: {correct_predictions}")
print(f"   Batch accuracy: {batch_accuracy:.2%}")

# Risk score distribution
risk_scores = [result['risk_score'] for result in batch_results]
avg_risk = np.mean(risk_scores)
high_risk_count = sum(1 for score in risk_scores if score > 0.8)

print(f"   Average risk score: {avg_risk:.3f}")
print(f"   High-risk packets (>0.8): {high_risk_count}")

## 🏁 Step 15: Tutorial Summary and Next Steps

Congratulations! You've successfully built and deployed your first packet classification system using Vision Transformers.

In [ ]:
# Final summary
print("🎉 Tutorial 3 Complete! 🎉")
print("=" * 50)

print("\n✅ What you've accomplished:")
accomplishments = [
    "Built a complete packet classification pipeline",
    "Implemented multiple packet-to-image encoding methods",
    "Fine-tuned a pre-trained Vision Transformer",
    "Achieved comprehensive model evaluation",
    "Visualized attention mechanisms for interpretability",
    "Created a production-ready deployment wrapper",
    "Tested the model with real inference scenarios"
]

for i, item in enumerate(accomplishments, 1):
    print(f"   {i}. {item}")

print(f"\n📊 Final Model Performance:")
print(f"   Test Accuracy: {evaluation_results['accuracy']:.2%}")
print(f"   ROC AUC: {evaluation_results['roc_auc']:.3f}")
print(f"   Detection Rate: {evaluation_results['detection_rate']:.2%}")
print(f"   False Alarm Rate: {evaluation_results['false_alarm_rate']:.2%}")

print("\n🚀 Next Steps:")
next_steps = [
    "Tutorial 4: Model Interpretability and Explainability",
    "Advanced Tutorial: Few-Shot Learning for New Threats",
    "Research Guide: Custom Architecture Design",
    "Production Guide: Scalable Deployment Strategies",
    "Experiment with real packet capture (PCAP) files",
    "Try different ViT architectures (ViT-Large, DeiT, etc.)",
    "Implement online learning for evolving threats"
]

for i, item in enumerate(next_steps, 1):
    print(f"   {i}. {item}")

print("\n💡 Key Takeaways:")
takeaways = [
    "Vision Transformers can effectively classify network packets",
    "Packet-to-image encoding preserves spatial relationships",
    "Transfer learning significantly reduces training time",
    "Attention visualization provides model interpretability",
    "Proper evaluation metrics are crucial for security applications"
]

for item in takeaways:
    print(f"   • {item}")

print("\n🔗 Resources for Further Learning:")
resources = [
    "Hugging Face Transformers Documentation",
    "Vision Transformer Paper (Dosovitskiy et al.)",
    "Network Security and Malware Detection Research",
    "PyTorch Computer Vision Tutorials",
    "Our project GitHub repository"
]

for item in resources:
    print(f"   📚 {item}")

print("\n" + "=" * 50)
print("Thank you for completing Tutorial 3! 🙏")
print("Happy packet hunting! 🕵️‍♂️🔍")

## 📝 Exercise Solutions and Extensions

### Exercise 1: Custom Encoding Method
Try implementing a protocol-aware encoding that treats different protocol layers differently.

### Exercise 2: Data Augmentation
Implement data augmentation techniques specific to packet data (byte shifting, noise injection, etc.).

### Exercise 3: Model Ensemble
Create an ensemble of models using different encoding methods and combine their predictions.

### Exercise 4: Real-time Processing
Modify the classifier to process packets in real-time from a network stream.

---

**🎯 Tutorial Complete!** You now have a solid foundation in packet classification using Vision Transformers. Keep experimenting and building upon these concepts!